# 10 — Real World ML Project
**Referencias:** ESL Cap. 7 (model selection) · Géron Cap. 2, 3 (end-to-end)

## El workflow completo de ML (Géron Cap. 2)
1. **Define el problema** → métricas de negocio
2. **Obtén los datos** → EDA
3. **Preprocesa y transforma** → Pipeline
4. **Explora modelos** → CV rápido
5. **Afina el mejor** → GridSearch / Nested CV
6. **Análisis de errores** → entender dónde falla
7. **Calibra y evalúa** → test set final
8. **Documenta y despliega** → joblib + monitoreo

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.base import BaseEstimator, TransformerMixin
import joblib, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Imports OK')

## Paso 1 — Definir el problema
**Objetivo de negocio:** predecir qué usuarios activarán su tarjeta en los próximos 7 días.  
**Métrica de negocio:** Revenue por campaña de activación.  
**Métrica de ML:** AUC-ROC (discriminación) + calibración (probabilidades confiables para scoring).

In [ ]:
# Paso 2 — Simular datos reales con todos los desafíos típicos
n = 5000
df = pd.DataFrame({
    'sessions':          np.random.randint(1, 30, n).astype(float),
    'days_since_reg':    np.random.randint(0, 60, n).astype(float),
    'time_on_site_avg':  np.abs(np.random.normal(200, 100, n)),
    'pages_per_session': np.abs(np.random.normal(3, 1.5, n)),
    'revenue_30d':       np.abs(np.random.lognormal(4, 1, n)),
    'support_calls':     np.random.poisson(0.8, n).astype(float),
    'login_errors':      np.random.poisson(0.3, n).astype(float),
    'channel':           np.random.choice(['organic','paid','email','direct','referral'], n),
    'device':            np.random.choice(['mobile','desktop','tablet'], n),
    'plan':              np.random.choice(['free','basic','pro'], n),
    'country':           np.random.choice(['CL','PE','CO','MX','AR'], n),
})

# Missing realistas
for col in ['time_on_site_avg','revenue_30d','support_calls']:
    mask = np.random.choice(n, size=int(n*0.07), replace=False)
    df.loc[mask, col] = np.nan

# Outliers en revenue
outlier_idx = np.random.choice(n, 30)
df.loc[outlier_idx, 'revenue_30d'] *= 50

# Target con efecto real
logit = (-2
    + df['sessions'].fillna(0)*0.08
    + df['pages_per_session'].fillna(3)*0.2
    - df['days_since_reg'].fillna(30)*0.04
    + np.log1p(df['revenue_30d'].fillna(0))*0.15
    - df['support_calls'].fillna(0)*0.3
    + (df['plan'] == 'pro').astype(float)*0.5
)
df['activated'] = (np.random.uniform(0,1,n) < 1/(1+np.exp(-logit))).astype(int)

num_feats = ['sessions','days_since_reg','time_on_site_avg','pages_per_session',
             'revenue_30d','support_calls','login_errors']
cat_feats  = ['channel','device','plan','country']

X = df[num_feats + cat_feats]
y = df['activated']

# Split: train / val / test (60/20/20)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Activation rate: train={y_train.mean():.2%}, val={y_val.mean():.2%}, test={y_test.mean():.2%}')

## Paso 3 — EDA rápido antes de modelar

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, col in enumerate(num_feats):
    ax = axes[i//4][i%4]
    for cls, color in [(0,'#adb5bd'),(1,'#4361ee')]:
        vals = df.loc[df['activated']==cls, col].dropna()
        ax.hist(vals, bins=25, alpha=0.6, color=color, density=True, label=f'Act={cls}')
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)

# Último subplot: target balance
axes[1][3].bar(['No activó','Activó'], [1-y.mean(), y.mean()], color=['#adb5bd','#4361ee'])
axes[1][3].set_title(f'Target balance\n{y.mean():.1%} activación')

plt.suptitle('EDA — Distribución por clase', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Paso 4 — Pipeline con feature engineering

In [ ]:
class UserBehaviorTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X['engagement_score'] = (
            X['sessions'].fillna(0) * X['pages_per_session'].fillna(3) /
            (X['days_since_reg'].fillna(30) + 1)
        ).clip(0, 100)
        X['revenue_per_session'] = (
            np.log1p(X['revenue_30d'].fillna(0)) / (X['sessions'].fillna(1) + 1)
        ).clip(0, 50)
        X['friction_score'] = X['support_calls'].fillna(0) + X['login_errors'].fillna(0)*0.5
        return X


extended_num = num_feats + ['engagement_score','revenue_per_session','friction_score']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imp', KNNImputer(n_neighbors=5)),
        ('sc',  RobustScaler()),
    ]), extended_num),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), cat_feats),
])

def build_pipeline(model):
    return Pipeline([
        ('feat_eng', UserBehaviorTransformer()),
        ('prep',     preprocessor),
        ('model',    model),
    ])

print('Pipeline construido')

## Paso 5 — Explorar modelos con CV rápido

In [ ]:
candidates = [
    ('Logistic Regression', LogisticRegression(C=0.1, max_iter=1000)),
    ('Random Forest',       RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)),
    ('Gradient Boosting',   GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
exploration = []
for name, model in candidates:
    pipe = build_pipeline(model)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    exploration.append({'model': name, 'mean': scores.mean(), 'std': scores.std()})
    print(f'{name:25s}: AUC {scores.mean():.4f} ± {scores.std():.4f}')

best_name = max(exploration, key=lambda x: x['mean'])['model']
print(f'\nMejor candidato: {best_name}')

## Paso 6 — Afinar el mejor modelo

In [ ]:
gb_pipe = build_pipeline(
    GradientBoostingClassifier(random_state=42)
)

param_grid = {
    'model__n_estimators':   [100, 200],
    'model__learning_rate':  [0.05, 0.1],
    'model__max_depth':      [3, 4],
    'model__subsample':      [0.8, 1.0],
}

grid = GridSearchCV(
    gb_pipe, param_grid,
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='roc_auc', n_jobs=-1, verbose=0
)
grid.fit(X_train, y_train)

print('Mejores parámetros:')
for k, v in grid.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nVal AUC (CV): {grid.best_score_:.4f}')

## Paso 7 — Análisis de errores (Géron Cap. 3)

In [ ]:
best_pipe = grid.best_estimator_
y_pred    = best_pipe.predict(X_val)
y_proba   = best_pipe.predict_proba(X_val)[:,1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['No act','Activó']); axes[0].set_yticklabels(['No act','Activó'])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i,j]), ha='center', va='center',
                     fontsize=14, color='white' if cm[i,j] > cm.max()//2 else 'black')
axes[0].set_xlabel('Predicho'); axes[0].set_ylabel('Real')
axes[0].set_title('Confusion Matrix')

# ROC curve
RocCurveDisplay.from_predictions(y_val, y_proba, ax=axes[1], name='GBM')
axes[1].set_title('ROC Curve')

# Precision-Recall
PrecisionRecallDisplay.from_predictions(y_val, y_proba, ax=axes[2], name='GBM')
axes[2].set_title('Precision-Recall Curve')

plt.tight_layout()
plt.show()

# Análisis de errores: ¿dónde falla?
val_df = X_val.copy()
val_df['y_true'] = y_val.values
val_df['y_pred'] = y_pred
val_df['proba']  = y_proba

# Falsos negativos: activadores que no detectamos
fn = val_df[(val_df['y_true']==1) & (val_df['y_pred']==0)]
# Falsos positivos: no-activadores predichos como activadores
fp = val_df[(val_df['y_true']==0) & (val_df['y_pred']==1)]

print(f'Falsos Negativos: {len(fn)} (perdemos {len(fn)/y_val.sum():.1%} de activadores reales)')
print(f'Falsos Positivos: {len(fp)}')
print()
print('Perfil de Falsos Negativos (activadores que el modelo pierde):')
print(fn[num_feats[:5]].describe().round(1))

## Paso 8 — Calibración de probabilidades (Géron Cap. 3 / ESL)

In [ ]:
# GBM tiende a producir probabilidades extremas → calibrar
calibrated = CalibratedClassifierCV(best_pipe, cv='prefit', method='isotonic')
calibrated.fit(X_val, y_val)

y_proba_cal = calibrated.predict_proba(X_test)[:,1]
y_proba_raw = best_pipe.predict_proba(X_test)[:,1]

fig, ax = plt.subplots(figsize=(7, 6))
frac_raw, mean_raw = calibration_curve(y_test, y_proba_raw, n_bins=10)
frac_cal, mean_cal = calibration_curve(y_test, y_proba_cal, n_bins=10)

ax.plot([0,1],[0,1], '--', color='#adb5bd', label='Perfecta calibración')
ax.plot(mean_raw, frac_raw, 'o-', color='#f72585', linewidth=2, label='GBM sin calibrar')
ax.plot(mean_cal, frac_cal, 's-', color='#4361ee', linewidth=2, label='GBM calibrado (isotonic)')
ax.set_xlabel('Probabilidad predicha')
ax.set_ylabel('Fracción positivos reales')
ax.set_title('Calibration Curve — Géron Cap. 3')
ax.legend()
plt.tight_layout()
plt.show()

auc_raw = roc_auc_score(y_test, y_proba_raw)
auc_cal = roc_auc_score(y_test, y_proba_cal)
print(f'AUC sin calibrar: {auc_raw:.4f}')
print(f'AUC calibrado:    {auc_cal:.4f}')
print('La calibración mejora la confiabilidad de P(activación) para ranking/scoring')

## Paso 9 — Permutation importance + evaluación final en test

In [ ]:
perm = permutation_importance(
    best_pipe, X_test, y_test, n_repeats=15, scoring='roc_auc', random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance': perm.importances_mean,
    'std': perm.importances_std,
}).sort_values('importance', ascending=False).head(10)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

perm_sorted = perm_df.sort_values('importance')
axes[0].barh(perm_sorted['feature'], perm_sorted['importance'],
             xerr=perm_sorted['std'], color='#4361ee', alpha=0.85, capsize=4)
axes[0].axvline(0, color='#888', linewidth=0.8)
axes[0].set_title('Permutation Importance — Top 10 (en test)')

# Distribución de probabilidades por segmento
test_df_plot = X_test.copy()
test_df_plot['proba'] = y_proba_cal
test_df_plot['activated'] = y_test.values
for plan, color in zip(['free','basic','pro'], ['#adb5bd','#4361ee','#f72585']):
    mask = test_df_plot['plan'] == plan
    axes[1].hist(test_df_plot.loc[mask, 'proba'], bins=25, alpha=0.6,
                 color=color, density=True, label=plan)
axes[1].set_xlabel('P(activación)')
axes[1].set_title('Distribución de P(activación) por Plan')
axes[1].legend()

plt.tight_layout()
plt.show()

print('=== EVALUACIÓN FINAL EN TEST SET ===')
print(f'AUC:      {auc_cal:.4f}')
print(f'\n{classification_report(y_test, calibrated.predict(X_test))}')

## Paso 10 — Deployment y monitoreo de data drift

In [ ]:
# Guardar modelo final
joblib.dump(calibrated, 'data/model_activation_v1.pkl', compress=3)

# Simular monitoreo de data drift (PSI — Population Stability Index)
def psi(expected, actual, n_bins=10):
    """PSI > 0.25 indica drift significativo."""
    bins = np.linspace(0, 1, n_bins+1)
    exp_pct  = np.histogram(expected, bins)[0] / len(expected) + 1e-8
    act_pct  = np.histogram(actual,   bins)[0] / len(actual)   + 1e-8
    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

# Simular producción: semana con drift en sessions
prod_normal = X_test.copy()
prod_drift  = X_test.copy()
prod_drift['sessions'] = prod_drift['sessions'] * 2 + np.random.normal(0, 5, len(prod_drift))
prod_drift['sessions'] = prod_drift['sessions'].clip(1, 50)

for col in ['sessions','days_since_reg','pages_per_session']:
    psi_normal = psi(X_train[col].dropna(), prod_normal[col].dropna())
    psi_drift  = psi(X_train[col].dropna(), prod_drift[col].dropna())
    status_n = 'OK' if psi_normal < 0.1 else ('WARN' if psi_normal < 0.25 else 'DRIFT')
    status_d = 'OK' if psi_drift  < 0.1 else ('WARN' if psi_drift  < 0.25 else 'DRIFT')
    print(f'{col:25s}: normal PSI={psi_normal:.3f} [{status_n}] | drift PSI={psi_drift:.3f} [{status_d}]')

import os
print(f'\nModelo guardado: {os.path.getsize("data/model_activation_v1.pkl")/1024:.0f} KB')
print('\nPSI thresholds: < 0.10 = sin cambio, 0.10-0.25 = cambio moderado, > 0.25 = drift significativo')

## Resumen del Curso

| Notebook | Tema | Conceptos clave |
|---|---|---|
| 01 | Foundations | EPE, Bias-Variance, NFL (ESL 2) |
| 02 | Preprocessing | RobustScaler, KNNImputer, Custom Transformer (Géron 2) |
| 03 | Regression | OLS, Ridge, Lasso, ElasticNet, coeff paths (ESL 3) |
| 04 | Classification | Bayes, LDA/QDA, calibración, imbalanced (ESL 4) |
| 05 | Model Evaluation | Nested CV, AIC/BIC, permutation importance (ESL 7) |
| 06 | Clustering | K-Means, GMM+BIC, jerárquico, DBSCAN (ESL 14) |
| 07 | Dimensionality | PCA, Kernel PCA, IPCA, t-SNE, SelectFromModel (ESL 14.5) |
| 08 | Pipelines | ColumnTransformer, custom steps, caching, joblib (Géron 2) |
| 09 | Ensembles | Bagging/OOB, GB+early stopping, VotingClassifier, Stacking (ESL 10, 15) |
| 10 | Real World | EDA → Pipeline → Grid → Error Analysis → Calibración → PSI (Géron 2-3) |

**Próximo:** Databricks / PySpark en Community Edition para datos a escala.